# 03 · Variance reduction

Since the error is σ/√n, halving σ is worth quadrupling n. Variance reduction exploits what we know about the
problem: symmetry (antithetic variates), a correlated quantity with a known mean (control variates), where the
important outcomes lie (importance sampling), how the variance is spread over the input space (stratified
sampling), and - when comparing systems - the fact that both see the same randomness (common random numbers).

**What you will learn**
- apply antithetic variates and predict when they help or hurt
- build control-variate estimators with estimated optimal coefficients
- design an importance-sampling proposal for a rare event
- use stratification and common random numbers

**Before you start:** notebook 02  ·  **Time:** about 90 min

In [1]:
try:                      # on Google Colab (or anywhere engstoch is missing): install it from GitHub
    import engstoch
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engstoch"], check=True)
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from engstoch import montecarlo as mc, queues, special as sf, brownian as bm
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

R = np.random.default_rng      # R(seed): a fresh, reproducible generator

## Antithetic variates

In [2]:
rows = []
for name, h in (("e^u (monotone)", np.exp), ("u^2 (monotone on [0,1])", lambda u: u**2), ("sin(2 pi u)^2 (symmetric)", lambda u: np.sin(2 * np.pi * u) ** 2), ("(u - 0.5)^2 (even about 1/2)", lambda u: (u - 0.5) ** 2)):
    r = mc.antithetic(h, 50_000, R(1))
    rows.append((name, r["correlation"], r["variance_reduction"]))
print(pd.DataFrame(rows, columns=["integrand", "corr(h(U), h(1-U))", "variance reduction"]).to_string(index=False, float_format="%.3f"))

                   integrand  corr(h(U), h(1-U))  variance reduction
              e^u (monotone)              -0.968              30.969
     u^2 (monotone on [0,1])              -0.875               8.009
   sin(2 pi u)^2 (symmetric)               1.000               0.500
(u - 0.5)^2 (even about 1/2)               1.000               0.500


Pairing U with 1 − U costs nothing and helps exactly when h(U) and h(1 − U) are negatively correlated - always
for monotone h. For a function symmetric about ½ the pair is perfectly positively correlated and antithetics
*double* the variance per evaluation: the method must match the problem.

## Control variates

In [3]:
S0, K, r_, sig, T = 100.0, 100.0, 0.05, 0.2, 1.0
ST = bm.gbm(S0, r_, sig, T, 1, 100_000, R(2))[:, -1]
payoff = np.exp(-r_ * T) * np.maximum(ST - K, 0)
plain = mc.mean_ci(payoff)
cv = mc.control_variate(payoff, ST, S0 * np.exp(r_ * T))
print(f"European call: plain MC {plain['estimate']:.4f} +- {plain['std_error']:.4f}; control variate S_T: {cv['estimate']:.4f} +- {cv['std_error']:.4f}")
print(f"  beta* = {cv['beta']:.3f}, R^2 = {cv['r_squared']:.3f}, variance reduction {cv['variance_reduction']:.1f}x; Black-Scholes {bm.black_scholes_call(S0, K, r_, sig, T):.4f}")
two = mc.control_variate(payoff, np.column_stack([ST, np.maximum(ST - 110, 0)]), [S0 * np.exp(r_ * T), np.exp(r_ * T) * bm.black_scholes_call(S0, 110, r_, sig, T)])
print(f"  two controls (S_T and the K = 110 call): {two['estimate']:.4f} +- {two['std_error']:.5f}, variance reduction {two['variance_reduction']:.0f}x")

European call: plain MC 10.4188 +- 0.0464; control variate S_T: 10.4400 +- 0.0177
  beta* = 0.640, R^2 = 0.854, variance reduction 6.8x; Black-Scholes 10.4506
  two controls (S_T and the K = 110 call): 10.4525 +- 0.00702, variance reduction 44x


A control variate C with known mean is subtracted with the coefficient β* = Cov(Y, C)/Var(C) - estimated by a
regression of Y on C - and the variance falls by the factor 1/(1 − R²). The terminal stock price explains
85 % of the call's variance; a second, similar option with a known price explains almost 98 % of it. The
best controls are simplified versions of the problem with closed-form answers.

## Importance sampling

In [4]:
a = 4.5
p = 1 - sf.norm_cdf(a)
rows = []
for theta in (0.0, 2.0, 3.5, 4.5, 5.5, 8.0):
    r = mc.importance_sampling(lambda z: (z > a).astype(float), lambda z: -0.5 * z**2, lambda n, t=theta: R(3).normal(t, 1, n), lambda z, t=theta: -0.5 * (z - t) ** 2, 100_000)
    rows.append((theta, r["estimate"], mc.relative_error(r) if r["estimate"] > 0 else np.inf))
print(f"P(Z > {a}) = {p:.4e}")
print(pd.DataFrame(rows, columns=["proposal mean theta", "estimate", "relative error"]).to_string(index=False, float_format="%.3g"))

P(Z > 4.5) = 3.3977e-06
 proposal mean theta  estimate  relative error
                   0         0             inf
                   2  3.55e-06          0.0428
                 3.5  3.42e-06          0.0101
                 4.5  3.39e-06         0.00714
                 5.5  3.35e-06           0.009
                   8  2.93e-06           0.145


Sampling from N(θ, 1) and reweighting by the likelihood ratio e^{−θz + θ²/2} keeps the estimator unbiased
for every θ, but its variance depends dramatically on θ: θ = 0 is crude Monte Carlo (no hits at all), θ ≈ a
puts half the samples in the rare region and gives 1 % accuracy with 10⁵ samples, and θ too large wastes them
far in the tail. Exponential tilting to the "dominating point" is the standard choice; a poor proposal can give
infinite variance while *looking* fine.

## Stratified sampling

In [5]:
h = lambda u: np.where(u > 0.9, 10 * u, 0.1 * u)
plain_sd = np.std(h(R(4).random(1_000_000)))
for strata in (2, 10, 50):
    for alloc in ("proportional", "neyman"):
        r = mc.stratified(h, 10_000, strata, R(5), alloc)
        print(f"{strata:>3} strata, {alloc:<12}: estimate {r['estimate']:.5f} +- {r['std_error']:.5f}, variance reduction {mc.variance_reduction(r, plain_sd, 10_000):6.1f}x")
print("exact:", 0.1 * 0.81 / 2 + 10 * 0.19 / 2)

  2 strata, proportional: estimate 0.99708 +- 0.02676, variance reduction    1.1x
  2 strata, neyman      : estimate 0.98064 +- 0.01889, variance reduction    2.3x
 10 strata, proportional: estimate 0.99049 +- 0.00091, variance reduction  978.2x
 10 strata, neyman      : estimate 0.99052 +- 0.00031, variance reduction 8162.3x
 50 strata, proportional: estimate 0.99050 +- 0.00018, variance reduction 24365.6x
 50 strata, neyman      : estimate 0.99050 +- 0.00006, variance reduction 200571.7x
exact: 0.9904999999999999


Stratification removes the between-strata part of the variance. Proportional allocation can never hurt;
Neyman allocation (n_j ∝ p_j σ_j) puts samples where the variance is - here in the last tenth of the interval -
and multiplies the gain.

## Comparing systems: common random numbers

In [6]:
def mean_wait(service_scale):
    return lambda r: np.mean(queues.lindley(r.exponential(1 / 0.8, 2000), r.exponential(service_scale, 2000))[200:])
res = mc.common_random_numbers(mean_wait(1.0), mean_wait(0.95), 200)
print(f"mean wait reduction from a 5 % faster server: CRN {res['crn']['estimate']:.3f} +- {res['crn']['std_error']:.3f}, independent streams {res['independent']['estimate']:.3f} +- {res['independent']['std_error']:.3f}")
print(f"variance reduction {res['variance_reduction']:.0f}x - the same 200 replications, only the seeds differ")

mean wait reduction from a 5 % faster server: CRN 0.976 +- 0.027, independent streams 1.061 +- 0.093
variance reduction 12x - the same 200 replications, only the seeds differ


To compare two configurations, drive both with the same random numbers: the difference then reflects the
configurations rather than the luck of the draws. For a queue - whose waits are monotone in the service times
- the variance of the difference falls by an order of magnitude, and the independent-streams interval is too
wide even to be sure of the effect's size. CRN needs synchronisation (the same uniform used for the same
purpose in both runs), which is why simulations dedicate one stream per source of randomness.

## Inside the algorithm: the optimal control-variate coefficient

In [7]:
Y, C = payoff, ST
beta = np.cov(Y, C)[0, 1] / np.var(C, ddof=1)
adj = Y - beta * (C - S0 * np.exp(r_ * T))
print(f"beta by hand {beta:.4f} (library {cv['beta']:.4f}); estimate {adj.mean():.4f}; variance ratio {np.var(Y) / np.var(adj):.2f} = 1/(1 - rho^2) = {1 / (1 - np.corrcoef(Y, C)[0, 1] ** 2):.2f}")

beta by hand 0.6400 (library 0.6400); estimate 10.4400; variance ratio 6.85 = 1/(1 - rho^2) = 6.85


## Exercises
1. Price an Asian (arithmetic-average) call on GBM with 12 monitoring dates, using the geometric-average Asian call (closed form: a lognormal) as a control variate. Report the variance reduction.
2. Estimate P(S_100 > 30) for a sum of 100 Exp(1) variables minus 100 (i.e. P(sum > 130)) by exponential tilting of each summand. Find the best tilt and compare with the gamma tail from SciPy.
3. **Implement it yourself:** write stratified sampling with Neyman allocation for E[h(U)] from scratch (pilot run, allocation, estimate and standard error), and reproduce the library's result for h(u) = 10u on u > 0.9.